# 📒 Notebook 4 — KNN (User-Based Collaborative Filtering) Model

**Algorithm:** KNNWithMeans (K-Nearest Neighbors) via the `Surprise` library

**How KNN works (simple explanation):**
- For a given user, find the K most similar users (based on rating history)
- Look at what those similar users rated highly
- Recommend those restaurants to our user
- 'WithMeans' = adjusts for the fact that some users always rate high or always rate low

**KNN vs SVD:**
- KNN is more interpretable ('users like you loved this place')
- SVD is generally more accurate but is a black box
- We combine both in Notebook 5

**Requires:** Run Notebook 2 first to generate `data/` folder

## 1. Install Dependencies

In [1]:
#!pip install scikit-surprise pandas tqdm -q

## 2. Load Data

In [1]:
import pandas as pd
import pickle

train_df      = pd.read_csv("data/ratings_train.csv")
test_df       = pd.read_csv("data/ratings_test.csv")
business_meta = pd.read_csv("data/business_meta.csv")

with open("data/user_encoder.pkl", "rb") as f:
    user_encoder = pickle.load(f)
with open("data/business_encoder.pkl", "rb") as f:
    biz_encoder = pickle.load(f)

print(f"✅ Train: {len(train_df):,}  |  Test: {len(test_df):,}")
print(f"   Unique users     : {train_df['user_id'].nunique():,}")
print(f"   Unique businesses: {train_df['business_id'].nunique():,}")

✅ Train: 1,579,941  |  Test: 394,986
   Unique users     : 77,443
   Unique businesses: 33,016


## 3. Build Surprise Dataset

In [ ]:
from surprise import Dataset, Reader, KNNWithMeans
from surprise import accuracy

reader = Reader(rating_scale=(1, 5))

train_data = Dataset.load_from_df(
    train_df[["user_id", "business_id", "target_rating"]],
    reader
)
trainset = train_data.build_full_trainset()

print(f"✅ Surprise trainset built")
print(f"   Users  : {trainset.n_users:,}")
print(f"   Items  : {trainset.n_items:,}")
print(f"   Ratings: {trainset.n_ratings:,}")

✅ Surprise trainset built
   Users  : 77,443
   Items  : 33,016
   Ratings: 1,579,941


: 

## 4. Train KNNWithMeans Model

In [ ]:
import time

# KNN hyperparameters
# k         : number of neighbors to consider
# sim_options:
#   name    : similarity metric — 'pearson_baseline' is best for ratings
#   user_based: True = compare users, False = compare items (item-based CF)
#               User-based: 'people like you liked this'
#               Item-based:  'people who liked X also liked Y'
#   We use user-based here for interpretability

knn_model = KNNWithMeans(
    k=40,
    min_k=3,
    sim_options={
        "name":       "pearson_baseline",
        "user_based": True
    },
    verbose=True
)

print("🏋️ Training KNN model (computing user similarities — may take a few minutes)...")
start = time.time()
knn_model.fit(trainset)
elapsed = time.time() - start
print(f"✅ Training complete in {elapsed:.1f}s")

🏋️ Training KNN model (computing user similarities — may take a few minutes)...
Estimating biases using als...


## 5. Evaluate on Test Set

In [ ]:
testset = list(zip(
    test_df["user_id"].tolist(),
    test_df["business_id"].tolist(),
    test_df["target_rating"].tolist()
))

predictions = knn_model.test(testset)

rmse = accuracy.rmse(predictions, verbose=False)
mae  = accuracy.mae(predictions,  verbose=False)

print("=" * 40)
print("KNN Model Evaluation")
print("=" * 40)
print(f"RMSE : {rmse:.4f}  (lower is better, target < 1.0)")
print(f"MAE  : {mae:.4f}   (mean absolute error in stars)")

# Sample predictions
print("\nSample predictions (actual vs predicted):")
for pred in predictions[:5]:
    print(f"  User: {pred.uid[:8]}... | Biz: {pred.iid[:8]}... | "
          f"Actual: {pred.r_ui} | Predicted: {pred.est:.2f}")

KNN Model Evaluation
RMSE : 1.2532  (lower is better, target < 1.0)
MAE  : 0.9276   (mean absolute error in stars)

Sample predictions (actual vs predicted):
  User: S8oIpAi2... | Biz: o6S4BEyd... | Actual: 4 | Predicted: 2.33
  User: U7L73-H4... | Biz: 8PlUYO_9... | Actual: 4 | Predicted: 4.80
  User: FUUEU3ZN... | Biz: jNxvJeAU... | Actual: 4 | Predicted: 3.34
  User: AHRrG3T1... | Biz: _P6AQUby... | Actual: 5 | Predicted: 4.21
  User: IyIsFkVY... | Biz: XEQp7kQV... | Actual: 4 | Predicted: 3.79


## 6. Compare SVD vs KNN Metrics

In [ ]:
import os

# Load SVD metrics if available
print("=" * 50)
print("Model Comparison")
print("=" * 50)
print(f"{'Model':<10} {'RMSE':>8} {'MAE':>8}")
print("-" * 30)
print(f"{'KNN':<10} {rmse:>8.4f} {mae:>8.4f}")
print("\n(Compare with SVD metrics from Notebook 3)")
print("Lower RMSE/MAE = better accuracy")
print("Both models are used together in Notebook 5 (Hybrid)")

Model Comparison
Model          RMSE      MAE
------------------------------
KNN          1.2532   0.9276

(Compare with SVD metrics from Notebook 3)
Lower RMSE/MAE = better accuracy
Both models are used together in Notebook 5 (Hybrid)


## 7. Save KNN Model

In [ ]:
import pickle, os
os.makedirs("models", exist_ok=True)

with open("models/knn_model.pkl", "wb") as f:
    pickle.dump(knn_model, f)

print("✅ KNN model saved to models/knn_model.pkl")

✅ KNN model saved to models/knn_model.pkl


## 8. Test: Get Top-3 Recommendations for an Existing User

In [ ]:
def knn_recommend(user_id, city=None, state=None, top_k=3):
    """
    Get top-K restaurant recommendations for an existing user using KNN.
    
    Args:
        user_id : The user's string ID from the database
        city    : Optional city filter (e.g. 'Philadelphia')
        state   : Optional state filter (e.g. 'PA')
        top_k   : Number of recommendations to return
    """
    if user_id not in user_encoder["str2idx"]:
        print(f"⚠️  User '{user_id}' not found in training data. Use Notebook 5 for new users.")
        return []

    # Restaurants already rated by this user
    rated_biz = set(train_df[train_df["user_id"] == user_id]["business_id"].tolist())

    # Filter candidates by location
    candidates = business_meta.copy()
    if city:
        candidates = candidates[candidates["city"].str.lower() == city.lower()]
    if state:
        candidates = candidates[candidates["state"].str.upper() == state.upper()]

    candidates = candidates[~candidates["business_id"].isin(rated_biz)]

    if candidates.empty:
        print(f"⚠️  No unrated businesses found for city={city}, state={state}")
        return []

    # Predict ratings
    preds = []
    for _, biz in candidates.iterrows():
        pred = knn_model.predict(user_id, biz["business_id"])
        preds.append({
            "business_id":   biz["business_id"],
            "business_name": biz["business_name"],
            "city":          biz["city"],
            "state":         biz["state"],
            "avg_stars":     biz["business_avg_stars"],
            "categories":    biz["categories"],
            "knn_score":     pred.est
        })

    preds_df = pd.DataFrame(preds).sort_values("knn_score", ascending=False).head(top_k)

    print(f"\n🎯 KNN Top-{top_k} for user '{user_id[:12]}...' in {city or 'any city'}, {state or 'any state'}")
    print("─" * 65)
    for i, (_, row) in enumerate(preds_df.iterrows()):
        print(f"#{i+1} {row['business_name']} ({row['city']}, {row['state']})")
        print(f"   Categories : {row['categories']}")
        print(f"   Avg Stars  : ⭐{row['avg_stars']}")
        print(f"   KNN Score  : {row['knn_score']:.3f}")
    return preds_df


# Test with real user
sample_user = train_df["user_id"].iloc[0]
print(f"Testing with user: {sample_user}")
knn_recommend(sample_user, top_k=3)

Testing with user: L0r4HOIObbYhgpYXCCLVsQ

🎯 KNN Top-3 for user 'L0r4HOIObbYh...' in any city, any state
─────────────────────────────────────────────────────────────────
#1 Farmicia (Philadelphia, PA)
   Categories : Breakfast & Brunch, American (New), Mexican, French, Restaurants, Italian
   Avg Stars  : ⭐4.0
   KNN Score  : 4.000
#2 The Angelo Pizza (Philadelphia, PA)
   Categories : Restaurants, Italian, Pizza
   Avg Stars  : ⭐4.5
   KNN Score  : 4.000
#3 Romano's Macaroni Grill (Tampa, FL)
   Categories : Restaurants, Italian
   Avg Stars  : ⭐3.0
   KNN Score  : 4.000


,business_id,business_name,city,state,avg_stars,categories,knn_score
0,fOhnSqmO4XY5vSI8whVKSA,Farmicia,Philadelphia,PA,4.0,"Breakfast & Brunch, American (New), Mexican, F...",4.0
11835,nmeGoVi_U-bbnojJsi52Aw,The Angelo Pizza,Philadelphia,PA,4.5,"Restaurants, Italian, Pizza",4.0
11841,2pXS9GDt2csyLRn6rQkR-w,Romano's Macaroni Grill,Tampa,FL,3.0,"Restaurants, Italian",4.0


## 9. Test with City/State Filter

In [ ]:
sample_city  = business_meta["city"].value_counts().index[0]
sample_state = business_meta[business_meta["city"] == sample_city]["state"].iloc[0]
print(f"Testing city filter: {sample_city}, {sample_state}")

knn_recommend(sample_user, city=sample_city, state=sample_state, top_k=3)

Testing city filter: Philadelphia, PA

🎯 KNN Top-3 for user 'L0r4HOIObbYh...' in Philadelphia, PA
─────────────────────────────────────────────────────────────────
#1 Farmicia (Philadelphia, PA)
   Categories : Breakfast & Brunch, American (New), Mexican, French, Restaurants, Italian
   Avg Stars  : ⭐4.0
   KNN Score  : 4.000
#2 Wok Chinese Restaurant (Philadelphia, PA)
   Categories : Chinese, Restaurants, Seafood, Asian Fusion
   Avg Stars  : ⭐3.5
   KNN Score  : 4.000
#3 Asia Supermarket (Philadelphia, PA)
   Categories : Food, Chinese, Grocery, Cantonese, Restaurants
   Avg Stars  : ⭐3.5
   KNN Score  : 4.000


,business_id,business_name,city,state,avg_stars,categories,knn_score
0,fOhnSqmO4XY5vSI8whVKSA,Farmicia,Philadelphia,PA,4.0,"Breakfast & Brunch, American (New), Mexican, F...",4.0
1595,kqBLD5rueekc26t2HLI4AQ,Wok Chinese Restaurant,Philadelphia,PA,3.5,"Chinese, Restaurants, Seafood, Asian Fusion",4.0
1597,Vj28PSL29SUCUzGfkZpaDg,Asia Supermarket,Philadelphia,PA,3.5,"Food, Chinese, Grocery, Cantonese, Restaurants",4.0
